In [3]:
import torch
import torch.nn as nn
torch.manual_seed(42)

# MSE

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

In [9]:
# 예측값과 정답값 (회귀)
pred = torch.tensor([2.5, 5.0, 7.5])
target = torch.tensor([3.0, 5.0, 7.0])

# pytorch에서 제공하는 MSELoss 함수
mse_buildin = nn.MSELoss()(pred, target)

# 수식기반 수작업
mse_manual = torch.mean((pred - target) ** 2)
print(f"mse_buildin: {mse_buildin}")
print(f"mse_manual: {mse_manual}")

mse_buildin: 0.1666666716337204
mse_manual: 0.1666666716337204


# BCE

$$
BCE = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)\right]
$$

In [4]:
# 단일 샘플
# y = 1 일 때, BCE = -log(y_hat)
# 정답 확률을 높게 줄수록 손실이 낮아지는 형태가 된다.

y = torch.tensor([1.0])
yhat_good = torch.tensor([0.9]) # 정답 확률이 높음
yhat_bad = torch.tensor([0.1]) # 정답 확률이 낮음

# 수작업 계산
bce_good_manual = -(y*torch.log(yhat_good) + (1-y)*torch.log(1-yhat_good))
bce_bad_manual = -(y*torch.log(yhat_bad) + (1-y)*torch.log(1-yhat_bad))

# 내장된 BCE
bce_good_builtin = nn.BCELoss()(yhat_good, y)
bce_bad_builtin = nn.BCELoss()(yhat_bad, y)

print(f'bce_good_manual : {bce_good_manual}')
print(f'bce_bad_manual : {bce_bad_manual}')
print(f'bce_good_builtin : {bce_good_builtin}')
print(f'bce_bad_builtin : {bce_bad_builtin}')

bce_good_manual : tensor([0.1054])
bce_bad_manual : tensor([2.3026])
bce_good_builtin : 0.10536054521799088
bce_bad_builtin : 2.3025851249694824


In [32]:
# BCE 배치계산

prob_pred = torch.tensor([0.9, 0.2, 0.4])
label = torch.tensor([1.0, 0.0, 1.0])

nn.BCELoss()(prob_pred, label)

tensor(0.4149)

# CE
다중 분류 손실

$$
CE = -\sum_{i=1}^{m} y_i\log(\hat{y}_i)
$$

In [8]:
# 클래스 3개 [고양이, 강아지, 토끼] 정답이 강아지 (index = 1)
target_index = torch.tensor([1])
one_hot = torch.tensor([0.0, 1.0, 0.0]) # 원 - 핫 인코딩된 정답

# 좋은 예측과 나쁜 예측
p_good = torch.tensor([0.2, 0.7, 0.1])
p_bad = torch.tensor([0.7, 0.2, 0.1])

# 수작업
ce_good_manual = -(one_hot*torch.log(p_good)).sum()
ce_bad_manual = -(one_hot*torch.log(p_bad)).sum()

# torch bce logists 입력 log(p)
ce_loss = nn.CrossEntropyLoss()
ce_good_builtin = ce_loss(torch.log(p_good).unsqueeze(0), target_index)
ce_bad_builtin = ce_loss(torch.log(p_bad).unsqueeze(0), target_index)

print(ce_good_manual, ce_good_builtin)
print(ce_bad_manual, ce_bad_builtin)


tensor(0.3567) tensor(0.3567)
tensor(1.6094) tensor(1.6094)


In [31]:
from sklearn.datasets import load_diabetes # 회귀
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
torch.manual_seed(42)

X, y = load_diabetes(return_X_y = True)

scaler = StandardScaler()

x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=42)

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype = torch.float32)
x_test_t = torch.tensor(x_test, dtype = torch.float32)
# x_test = torch.tensor(x_test, dtype = torch.float32)

y_train_t = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype = torch.float32).unsqueeze(1)

print(x_train_t.shape)
reg_model = nn.Sequential(
    nn.Linear(x_train_t.shape[1], 64),
    nn.ReLU(), # 활성화 함수 통과시키기
    nn.Linear(64, 1)
)

mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=1e-2) # 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad() # 이전 가중치를 초기화
    pred = reg_model(x_train_t)
    loss = mse_loss(pred, y_train_t)
    loss.backward() # 가중치 찾아서 각 계산과정에 배치
    optimizer.step() # 각 계산과정에 가중치와 바이어스를 업데이트

with torch.no_grad():
    test_pred = reg_model(x_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

print(f'test mse : {test_mse.item():.4f}')

from sklearn.metrics import r2_score
r2_score(y_test_t, test_pred)    

torch.Size([331, 10])
test mse : 2997.9246


0.4578494429588318

### BCE 실습

In [ ]:
# 암환자인지 아닌지 여부

from sklearn.datasets import load_breast_cancer

### CE 실습